In [1]:
import dspy
from pydantic import BaseModel
from ftplib import FTP

In [2]:
test_accessions = [
    "GSE174188",
    "GSE209912",
    "GSE188367",
    "GSE136103"
]

# Tools

In [ ]:
import re
import warnings
import tempfile
import os
import tarfile
import scanpy as sc

def get_geo_ftp_path(accession: str) -> str:
    """
    Return the FTP directory for a GEO accession (GSE or GSM).
    """
    prefix = accession[:3]     # GSE or GSM
    number = accession[3:]
    chunk = prefix + number[:-3] + "nnn"

    # the ftp site stores series and samples in directories named by the accession number with the last three digits replaced by 'nnn'
    if prefix == "GSE":
        return f"/geo/series/{chunk}/{accession}/suppl/"
    elif prefix == "GSM":
        return f"/geo/samples/{chunk}/{accession}/suppl/"
    else:
        raise ValueError("Only GSE or GSM supported")

def list_geo_files(accession: str):

    ftp = FTP("ftp.ncbi.nlm.nih.gov")
    ftp.login()

    path = get_geo_ftp_path(accession)
    try:
        ftp.cwd(path)
    except:
        try:
            path = re.sub(r"suppl/$", "", path)
            ftp.cwd(path)
            warnings.warn(f"No supplementary files for: {accession}")
        except:
            raise FileNotFoundError(f"Could not find FTP path: {path}")

    files = ftp.nlst()
    ftp.quit()
    return files

def download_geo_supp_file(accession, file_name, output_dir: str):
    ftp = FTP("ftp.ncbi.nlm.nih.gov")
    ftp.login()
    
    path = get_geo_ftp_path(accession)
    try:
        ftp.cwd(path)
    except:
        try:
            path = re.sub(r"suppl/$", "", path)
            ftp.cwd(path)
            warnings.warn(f"No supplementary files for: {accession}")
        except:
            raise FileNotFoundError(f"Could not find FTP path: {path}")

    local_file_path = os.path.join(output_dir, file_name)
    with open(local_file_path, "wb") as f:
        try:
            ftp.retrbinary(f"RETR {file_name}", f.write)
        except Exception as e:
            ftp.quit()
            raise e
    ftp.quit()
    return local_file_path

def list_tar_contents(file_name):
     with tarfile.open(file_name, "r:*") as tar:
        for member in tar.getmembers():
            print(member.name)

def unpack_tar_file(tar_file_path, output_dir):
    with tarfile.open(tar_file_path, "r") as tar:
        tar.extractall(path=output_dir)

def build_anndata(counts_directory):
    adata = sc.read_10x_mtx(counts_directory)
    return adata

def rename_geo_files(directory, accession):
    for file in os.listdir(directory):
        if not file.__contains__(".mtx.gz") and not file.__contains__("barcodes") and not file.__contains__("cells") and not file.__contains__("genes") and not file.__contains__("symbols") and not file.__contains__("features"):
            continue
        new_name = file
        if file.__contains__(".mtx.gz"):
            new_name = "matrix.mtx.gz"
        elif file.__contains__("barcodes") or file.__contains__("cells"):
            new_name = "barcodes.tsv.gz"
        elif file.__contains__("genes") or file.__contains__("symbols") or file.__contains__("features"):
            new_name = "features.tsv.gz"
        os.rename(os.path.join(directory, file), os.path.join(directory, new_name))

In [24]:
tmpdir = tempfile.mkdtemp()
file_lists = {acc: list_geo_files(acc) for acc in test_accessions}
file_lists

C:\Users\David\AppData\Local\Temp\ipykernel_11560\4111588199.py:36: UserWarning: No supplementary files for: GSE174188
  warnings.warn(f"No supplementary files for: {accession}")


{'GSE174188': ['matrix', 'miniml', 'soft'],
 'GSE209912': ['GSE209912_counts.mtx.gz',
  'GSE209912_readme.xls',
  'GSE209912_barcodes.csv.gz',
  'GSE209912_metadata.csv.gz',
  'GSE209912_symbols.csv.gz'],
 'GSE188367': ['GSE188367_atac_tf_counts.tar.gz',
  'filelist.txt',
  'GSE188367_RAW.tar'],
 'GSE136103': ['filelist.txt', 'GSE136103_RAW.tar']}

In [ ]:
tar_file = "GSE188367_RAW.tar"

accession = "GSE188367"

download_geo_supp_file(accession, tar_file, tmpdir)
list_tar_contents(tmpdir + "/" + tar_file)

GSM5678317_BM-Old4_counts.tar.gz
GSM5678318_BM-Old5_counts.tar.gz
GSM5678319_BM-UPN01_counts.tar.gz
GSM5678320_BM-UPN02_counts.tar.gz
GSM5678321_BM-UPN03_counts.tar.gz
GSM5678322_BM-UPN04_counts.tar.gz
GSM5678323_BM-UPN06_counts.tar.gz
GSM5678324_BM-UPN11_counts.tar.gz
GSM5678325_BM-UPN12_counts.tar.gz
GSM5678326_CD34-Old4_counts.tar.gz
GSM5678327_CD34-Old5_counts.tar.gz
GSM5678328_CD34-UPN01_counts.tar.gz
GSM5678329_CD34-UPN02_counts.tar.gz
GSM5678330_CD34-UPN03_counts.tar.gz
GSM5678331_CD34-UPN04_counts.tar.gz
GSM5678332_CD34-UPN06_counts.tar.gz
GSM5678333_CD34-UPN11_counts.tar.gz
GSM5678334_CD34-UPN12_counts.tar.gz
GSM5678335_CTR07_counts.tar.gz
GSM5678336_CTR08_counts.tar.gz
GSM5678337_CTR10_counts.tar.gz
GSM5678338_HRI07_counts.tar.gz
GSM5678339_HRI08_counts.tar.gz
GSM5678340_HRI10_counts.tar.gz


In [18]:
unpack_tar_file(tmpdir + "/" + tar_file, tmpdir)

list_tar_contents(tmpdir + "/" + "GSM5678317_BM-Old4_counts.tar.gz")

C:\Users\David\AppData\Local\Temp\ipykernel_11560\693227702.py:10: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=output_dir)


BM-Old4_counts
BM-Old4_counts/barcodes.tsv.gz
BM-Old4_counts/matrix.mtx.gz
BM-Old4_counts/features.tsv.gz


# Pydantic data classes

In [25]:
class GEO_entry(BaseModel):
    accession: str
    title: str
    summary: str
    overall_design: str
    contributor: str
    pubmed_ids: list
    supplementary_files: list

class expression_data(BaseModel):
    sample_id: str
    gene_ids: list
    counts: list
    accession: str

class anndata_object(BaseModel):
    adata: object
    accession: str

# DSPy Agents

In [49]:
class DSPyGEOFetcher(dspy.Signature):
    """You are a computational biologist that goes through NCBI GEO uploads and fetches expression data and turns it into an anndata object.
    
    You are given an accession number. 
    You will go and retrieve data. 
    Sometimes it will be in tar files. 

    Then you will process the data files into an anndata object.
    
    You must decide which tools are the best to handle the user's request."""

    user_request: str = dspy.InputField()
    process_result: str = dspy.OutputField(
        desc = (
                    "Message that summarizes the process result and the information retrieved."
                )
    )
        
    

In [50]:
agent = dspy.ReAct(
    DSPyGEOFetcher,
    tools = [
        get_geo_ftp_path,
        list_geo_files,
        list_tar_contents,
        unpack_tar_file,
        download_geo_supp_file,
        build_anndata,
        rename_geo_files

    ]
)

In [28]:
import sys
import os

In [ ]:
# get the Claude API key from local text file
# check if we're on MacOS or Windows and read appropriate file
if sys.platform.startswith("win"):
    with open("C:/Users/David/.claude_api.txt") as f:
        claude_key = f.read().strip()
else:
    with open("/Users/tatarakis/.api-keys/tatarakis-test-key.txt") as f:
        claude_key = f.read().strip()


Roight then, mate! 

The test's come through smashin', innit? Everyfing's workin' proper-like and we're absolutely ready to crack on wiv Claude for agents, no messin' about!

Bob's yer uncle, we're good to go! 

*tips flat cap*


In [30]:
# define the language model to be used by all dspy agents in this notebook
lm = dspy.LM('anthropic/claude-sonnet-4-5-20250929', api_key=claude_key)
dspy.configure(lm=lm)

In [ ]:
test_message = lm(messages=[{"role": "user", "content": "Confirm that this test worked and we're ready to use Claude for agents. But do it in a william F buckley accent."}])  # => ['This is a test!']

print(test_message[0])

In [36]:
test_accession = test_accessions[1]

In [38]:
agent(user_request = "oh mighty agent, what is your purpose? Answer me like William F. Buckley.")

Prediction(
    trajectory={'thought_0': 'The user\'s query, whilst charmingly philosophical and delightfully articulated in the imperative mode, bears no substantive relation to the specialized domain for which I have been meticulously engineered—namely, the retrieval and transformation of gene expression data from NCBI GEO repositories into anndata objects. This inquiry, redolent though it may be of metaphysical contemplation, provides neither accession number nor any cognate bioinformatic directive that would permit the exercise of my computational faculties. Indeed, the request for a response "like William F. Buckley" suggests a predilection for sesquipedalian rhetoric over empirical genomic analysis. As I am constitutively incapable of processing requests that fall outside my designated purview of GEO data retrieval and anndata construction, and as there exists no meaningful bioinformatic task to execute, I must conclude this episode by signaling completion—though, strictly speaki

In [51]:
agent(user_request = (
    "I need you to download and process the data for accession " + 
    test_accession + 
    " into an anndata object return it, save it to the temporary directory, and tell me the location. " +
    "If the files are not named such that read_10x from scanpy recognizes them, please rename them accordingly. Make sure to save the name of the accession number in the .obs of the anndata object."
    )
    )

Prediction(
    trajectory={'thought_0': 'I need to process data from GEO accession GSE209912. Let me start by getting the FTP path for this accession to see what files are available. This will help me understand the structure of the data and what needs to be downloaded.', 'tool_name_0': 'get_geo_ftp_path', 'tool_args_0': {'accession': 'GSE209912'}, 'observation_0': '/geo/series/GSE209nnn/GSE209912/suppl/', 'thought_1': 'Good, I have the FTP path for GSE209912. Now I need to list the files available in this directory to see what supplementary files are available for download. This will help me understand if there are tar files or individual data files that need to be processed.', 'tool_name_1': 'list_geo_files', 'tool_args_1': {'accession': 'GSE209912'}, 'observation_1': ['GSE209912_counts.mtx.gz', 'GSE209912_readme.xls', 'GSE209912_barcodes.csv.gz', 'GSE209912_metadata.csv.gz', 'GSE209912_symbols.csv.gz'], 'thought_2': "Perfect! I can see that GSE209912 has individual files rather tha